In [1]:
from langchain_huggingface import HuggingFaceEmbeddings

d:\LangChain-Concepts\LangChain-Vector-Stores\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
embedding = HuggingFaceEmbeddings(
    model_name = 'Qwen/Qwen3-Embedding-0.6B'
)

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main/./sentence_bert_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [ ]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [ ]:
# FAISS: create vector store directly from documents (index is held in memory)
# Unlike Chroma, FAISS doesn't persist automatically — you save/load manually

vector_store = FAISS.from_documents(
    documents = docs,
    embedding = embedding
)

# Save index to disk (equivalent of Chroma's persist_directory)
vector_store.save_local('faiss_db')

In [ ]:
# To reload a saved FAISS index in a later session:
# vector_store = FAISS.load_local('faiss_db', embeddings=embedding, allow_dangerous_deserialization=True)

In [ ]:
# Add more documents to an existing FAISS index
# any number of docs can be added

# vector_store.add_documents(docs)   # same API as Chroma

# Note: after adding, save again to persist:
# vector_store.save_local('faiss_db')

In [ ]:
# View documents stored in FAISS
# FAISS doesn't have a .get() method like Chroma.
# Access the internal docstore directly instead:

for doc_id, doc in vector_store.docstore._dict.items():
    print(f"ID: {doc_id}")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")
    print()

In [ ]:
# Searching

vector_store.similarity_search(
    query = "who among these are the bowlers?",
    k = 2   # how many similar results you want
)

In [ ]:
# Search with similarity score
# FAISS returns L2 distance (lower = more similar), same as Chroma

vector_store.similarity_search_with_score(
    query = 'who among those is a cricketer?',
    k = 2
)

In [ ]:
# Metadata filtering
# FAISS supports metadata filtering via the filter parameter (dict or callable)

vector_store.similarity_search_with_score(
    query = '',
    filter = {'team': 'Royal Challengers Bangalore'}
)

# Same semantic as Chroma: query similarity uses embeddings, metadata filter is an exact-match condition

In [ ]:
# Update a document
# FAISS has no direct update_document() method.
# Standard approach: delete by ID then re-add.

# 1. Find the internal FAISS index id for the doc you want to update
doc_id_to_update = None
for doc_id, doc in vector_store.docstore._dict.items():
    if 'Virat Kohli' in doc.page_content:
        doc_id_to_update = doc_id
        break

print(f"Found doc ID: {doc_id_to_update}")

# 2. Delete the old doc
if doc_id_to_update:
    vector_store.delete([doc_id_to_update])

# 3. Add the updated doc
updated_doc1 = Document(
    page_content = "Virat Kohli is former captain of Royal Challengers Bangalore (RCB). He is renowned for his consistency and aggressive leadership.",
    metadata = {'team': 'Royal Challengers Bangalore'}
)
vector_store.add_documents([updated_doc1])

# Persist changes
vector_store.save_local('faiss_db')

In [ ]:
# View documents after update

for doc_id, doc in vector_store.docstore._dict.items():
    print(f"ID: {doc_id}")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")
    print()

In [ ]:
# Delete a document by ID

doc_id_to_delete = None
for doc_id, doc in vector_store.docstore._dict.items():
    if 'Virat Kohli' in doc.page_content:
        doc_id_to_delete = doc_id
        break

if doc_id_to_delete:
    vector_store.delete([doc_id_to_delete])
    vector_store.save_local('faiss_db')
    print(f"Deleted doc: {doc_id_to_delete}")

In [ ]:
# View documents after deletion

for doc_id, doc in vector_store.docstore._dict.items():
    print(f"ID: {doc_id}")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")
    print()